# 🔬 DEBUG - Inspeção da Estrutura HTML do InHire

**Objetivo:** Descobrir a estrutura HTML REAL do site InHire para ajustar os seletores corretamente.

**Por que precisamos disso?**  
O código original não encontrou vagas porque os seletores CSS/HTML estavam genéricos demais.  
Vamos inspecionar o HTML real e criar seletores precisos!

In [ ]:
# Instalação (execute apenas uma vez)
!pip install requests beautifulsoup4 lxml --quiet

In [ ]:
import requests
from bs4 import BeautifulSoup
import re

# Headers para simular navegador
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7',
}

print("✅ Bibliotecas carregadas!")

---
## 🔍 TESTE 1: Inspecionar Sympla (sabemos que tem vaga)

Você encontrou uma vaga da Sympla no Google. Vamos inspecionar essa página!

In [ ]:
# URL da Sympla
url = "https://sympla.inhire.app/vagas"

print(f"🔍 Acessando: {url}\n")

try:
    response = requests.get(url, headers=headers, timeout=10)
    
    print(f"📊 Status Code: {response.status_code}")
    print(f"📏 Tamanho da resposta: {len(response.content)} bytes\n")
    
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'lxml')
        
        # Salva HTML completo para análise
        with open('sympla_inhire_debug.html', 'w', encoding='utf-8') as f:
            f.write(soup.prettify())
        
        print("✅ HTML salvo em: sympla_inhire_debug.html")
        print("\n" + "="*70)
        print("📄 PRIMEIROS 3000 CARACTERES DO HTML:")
        print("="*70)
        print(soup.prettify()[:3000])
        
    else:
        print(f"❌ Erro: Página retornou status {response.status_code}")
        
except Exception as e:
    print(f"❌ Erro ao acessar página: {str(e)}")

---
## 🔍 TESTE 2: Procurar Padrões Específicos

Vamos procurar por padrões comuns em sites de vaga.

In [ ]:
if response.status_code == 200:
    soup = BeautifulSoup(response.content, 'lxml')
    
    print("🔍 BUSCANDO PADRÕES COMUNS...\n")
    
    # 1. Procurar por classes com 'job', 'vaga', 'position'
    print("="*70)
    print("1️⃣ CLASSES com 'job', 'vaga' ou 'position':")
    print("="*70)
    
    all_classes = []
    for element in soup.find_all(class_=True):
        classes = element.get('class', [])
        for cls in classes:
            if any(word in cls.lower() for word in ['job', 'vaga', 'position', 'card', 'item']):
                if cls not in all_classes:
                    all_classes.append(cls)
    
    if all_classes:
        for cls in all_classes[:20]:  # Primeiras 20
            print(f"  • {cls}")
    else:
        print("  ⚠️  Nenhuma classe encontrada")
    
    # 2. Procurar por links com '/vaga' ou '/job'
    print("\n" + "="*70)
    print("2️⃣ LINKS com '/vaga' ou '/job':")
    print("="*70)
    
    links = soup.find_all('a', href=re.compile('vaga|job|position', re.I))
    
    if links:
        print(f"  ✅ Encontrados {len(links)} links!\n")
        for i, link in enumerate(links[:5], 1):  # Primeiros 5
            href = link.get('href', '')
            texto = link.get_text(strip=True)[:60]
            print(f"  Link {i}:")
            print(f"    Texto: {texto}")
            print(f"    Href: {href}\n")
    else:
        print("  ⚠️  Nenhum link encontrado")
    
    # 3. Procurar por elementos <article>, <li>, <div> que possam ser vagas
    print("\n" + "="*70)
    print("3️⃣ ESTRUTURA (articles, li, divs):")
    print("="*70)
    
    articles = soup.find_all('article')
    print(f"  Articles: {len(articles)}")
    
    lis = soup.find_all('li')
    print(f"  Li's: {len(lis)}")
    
    # 4. Procurar texto "Analista" ou "Remoto" na página
    print("\n" + "="*70)
    print("4️⃣ BUSCA DE TEXTO 'Analista' ou 'Remoto':")
    print("="*70)
    
    texto_completo = soup.get_text().lower()
    
    if 'analista' in texto_completo:
        print("  ✅ Palavra 'analista' ENCONTRADA na página")
        # Encontra contexto
        matches = re.finditer(r'.{0,50}analista.{0,50}', texto_completo, re.I)
        for i, match in enumerate(list(matches)[:3], 1):
            print(f"    Contexto {i}: ...{match.group().strip()}...")
    else:
        print("  ❌ Palavra 'analista' NÃO encontrada")
    
    if 'remoto' in texto_completo:
        print("\n  ✅ Palavra 'remoto' ENCONTRADA na página")
    else:
        print("\n  ❌ Palavra 'remoto' NÃO encontrada")
    
    # 5. Verificar se a página é SPA (Single Page Application)
    print("\n" + "="*70)
    print("5️⃣ VERIFICAÇÃO DE SPA (JavaScript):")
    print("="*70)
    
    scripts = soup.find_all('script')
    print(f"  Scripts encontrados: {len(scripts)}")
    
    # Procura por React, Vue, Angular
    texto_html = str(soup)
    
    spa_frameworks = {
        'React': 'react',
        'Vue': 'vue',
        'Angular': 'angular',
        'Next.js': 'next',
    }
    
    for framework, keyword in spa_frameworks.items():
        if keyword in texto_html.lower():
            print(f"  ⚠️  {framework} detectado! (Pode ser SPA - precisa JavaScript)")
    
    # 6. Procurar por elementos com ID
    print("\n" + "="*70)
    print("6️⃣ IDs RELEVANTES:")
    print("="*70)
    
    elementos_com_id = soup.find_all(id=True)
    ids_relevantes = []
    
    for el in elementos_com_id:
        element_id = el.get('id', '')
        if any(word in element_id.lower() for word in ['job', 'vaga', 'position', 'list', 'app', 'root']):
            ids_relevantes.append(element_id)
    
    if ids_relevantes:
        for element_id in ids_relevantes[:10]:
            print(f"  • {element_id}")
    else:
        print("  ⚠️  Nenhum ID relevante encontrado")

else:
    print("⚠️  Não foi possível analisar (página não carregou)")

---
## 🔍 TESTE 3: Análise Manual de Elemento Específico

Se você souber uma classe ou ID específico, teste aqui!

In [ ]:
# INSIRA AQUI a classe ou ID que você quer testar
# Exemplo: procurar por div com classe específica

if response.status_code == 200:
    soup = BeautifulSoup(response.content, 'lxml')
    
    # Teste diferentes seletores
    seletores_teste = [
        ('div[class*="job"]', 'Divs com "job" na classe'),
        ('div[class*="card"]', 'Divs com "card" na classe'),
        ('a[href*="/vagas/"]', 'Links com /vagas/ no href'),
        ('[data-testid*="job"]', 'Elementos com data-testid "job"'),
    ]
    
    print("🧪 TESTANDO SELETORES COMUNS:\n")
    
    for seletor, descricao in seletores_teste:
        elementos = soup.select(seletor)
        print(f"  {descricao}: {len(elementos)} encontrado(s)")
        
        if elementos:
            print(f"    Exemplo: {str(elementos[0])[:200]}...\n")

---
## 🔍 TESTE 4: Testar Outras Empresas

Vamos verificar se KPMG, Alura também têm a mesma estrutura.

In [ ]:
empresas_teste = ['kpmg', 'alura', 'ifood']

print("🧪 TESTANDO MÚLTIPLAS EMPRESAS:\n")

for empresa in empresas_teste:
    url = f"https://{empresa}.inhire.app/vagas"
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        
        print(f"\n{'='*70}")
        print(f"🏢 {empresa.upper()}")
        print(f"{'='*70}")
        print(f"  URL: {url}")
        print(f"  Status: {response.status_code}")
        print(f"  Tamanho: {len(response.content)} bytes")
        
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'lxml')
            
            # Busca rápida
            texto = soup.get_text().lower()
            
            # Conta ocorrências de palavras-chave
            count_vaga = texto.count('vaga')
            count_analista = texto.count('analista')
            count_remoto = texto.count('remoto')
            
            print(f"  Ocorrências 'vaga': {count_vaga}")
            print(f"  Ocorrências 'analista': {count_analista}")
            print(f"  Ocorrências 'remoto': {count_remoto}")
            
            # Busca links
            links_vaga = soup.find_all('a', href=re.compile('vaga', re.I))
            print(f"  Links com 'vaga': {len(links_vaga)}")
            
        else:
            print(f"  ❌ Página não acessível")
    
    except Exception as e:
        print(f"  ❌ Erro: {str(e)[:50]}")
    
    import time
    time.sleep(2)  # Delay entre requisições

---
## 💡 TESTE 5: Verificar se é SPA com JSON embarcado

Muitos sites modernos carregam dados via JavaScript. Vamos procurar JSON no HTML.

In [ ]:
if response.status_code == 200:
    print("🔍 PROCURANDO DADOS JSON EMBARCADOS...\n")
    
    soup = BeautifulSoup(response.content, 'lxml')
    
    # Procura por script tags com JSON
    scripts = soup.find_all('script')
    
    for i, script in enumerate(scripts, 1):
        script_content = script.string
        
        if script_content and ('{' in script_content or '[' in script_content):
            # Pode conter JSON
            if any(word in script_content.lower() for word in ['job', 'vaga', 'position']):
                print(f"\n{'='*70}")
                print(f"📜 SCRIPT {i} - Possível JSON com vagas:")
                print(f"{'='*70}")
                print(script_content[:1000])  # Primeiros 1000 caracteres
                print("\n[...truncado...]\n")
    
    # Procura por __NEXT_DATA__ (Next.js)
    next_data = soup.find('script', id='__NEXT_DATA__')
    if next_data:
        print("\n" + "="*70)
        print("🎯 NEXT.JS DETECTADO - Dados em __NEXT_DATA__:")
        print("="*70)
        print(next_data.string[:2000])
        
        # Tenta parsear JSON
        try:
            import json
            data = json.loads(next_data.string)
            print("\n✅ JSON parseado com sucesso!")
            print(f"Chaves principais: {list(data.keys())}")
        except:
            print("\n⚠️  Não foi possível parsear JSON")

---
## 📊 RESUMO E PRÓXIMOS PASSOS

**Com base nos resultados acima, você deve:**

1. **Se encontrou classes específicas:** Anote as classes e vamos criar seletores precisos
2. **Se é SPA (React/Next.js):** Precisaremos usar Selenium ou extrair JSON
3. **Se encontrou links:** Podemos seguir os links individuais
4. **Se encontrou JSON:** Podemos parsear direto do JSON (muito mais rápido!)

**Execute todas as células acima e me mostre os resultados!**